# Práctica 2

### Objetivo
El objetivo de esta práctica es que el alumno comprenda, interprete y modifique el modelo de cinemática directa e inversa d eun moanipulador serial.

#### Metas
- Que el alumno aplique un modelo cinemático, obteniendo las matrices de transformación y la postura del manipulador
- Que el alumno aplique un modelo cinemático inverso para calcular una trayectoria a partir de una posición actual hacia una posición final
- Que el alumno grafique y analice los resultados del modelo

#### Cuestionario Previo

##### *1. ¿Qué son las transformaciones homogéneas?*

Son matrices de 4×4 que representan en una sola operación la rotación y la traslación entre dos sistemas de coordenadas. En robótica se usan para describir la posición y orientación de cada eslabón del robot respecto al anterior, y para encadenar esas relaciones hasta llegar al efector final.

##### *2. ¿Qué nos permite obtener el modelo de cinemática directa?*

Permite conocer la posición y orientación del efector final del robot en el espacio, a partir de los valores conocidos de sus articulaciones. Es decir, dados los ángulos o desplazamientos de cada junta, se obtiene dónde está y cómo está orientada la herramienta del robot.

##### *3. ¿Qué nos permite obtener el modelo de cinemática inversa?*

Permite determinar los valores que deben tener las articulaciones para que el efector final alcance una posición y orientación deseadas. Es el problema inverso: se sabe dónde se quiere llegar y se calculan los movimientos necesarios para lograrlo. Puede tener múltiples soluciones o ninguna, dependiendo de la configuración del robot.

##### *4. ¿De qué formas se puede interpolar la trayectoria del efector final entre dos puntos?*

La trayectoria puede interpolarse en el espacio articular, trabajando directamente con los ángulos de las juntas, o en el espacio cartesiano, controlando la posición y orientación del efector en cada instante. En cuanto al tipo de función usada para generar los puntos intermedios, las principales opciones son interpolación lineal, polinomial cúbica, polinomial de quinto grado, splines cúbicas y el método trapezoidal (LSPB). Cada una ofrece distintos grados de suavidad en posición, velocidad y aceleración.

#### Desarrollo
##### 1. Planteamiento de la cinemática directa
En esta primera parte, se crearán las transformaciones homogéneas y el modelo de cinemática directa de un robot RRR, incluyendo la matriz del Jacobiano. Se recomienda usar Sympy para el planteamiento de las expresiones. Un diagrama del robot se muestra en la imagen:

![imagen1](img/p2_1.png)

** Considerar valores cualesquiera para las dimensiones de los eslabones y la posición inicial de las juntas

In [ ]:
#Librerías
from sympy import *
import matplotlib.pyplot as plt
import numpy as np

In [ ]:
#!/usr/bin/env python3

class Robot():
    def __init__(self, l=(0.3, 0.3, 0.3)):

        # Variables simbólicas
        th1, th2, th3 = symbols("theta_1 theta_2 theta_3")

        # -----------------------------
        # Transformaciones homogéneas
        # -----------------------------

        T_0_1 = self.tr_h(alpha=th1)

        T_1_2 = self.tr_h(
            x=l[0],
            alpha=th2
        )

        T_2_3 = self.tr_h(
            x=l[1],
            alpha=th3
        )

        T_3_p = self.tr_h(
            x=l[2]
        )

        # Cinemática directa
        T_0_p = simplify(
            T_0_1 * T_1_2 * T_2_3 * T_3_p
        )

        print("Matriz homogénea T_0_p:")
        pprint(T_0_p)

        # --------------------------------
        # Vector de postura del efector
        # --------------------------------

        xi_0_p = Matrix([
            T_0_p[0, 3],     # x
            T_0_p[1, 3],     # y
            th1 + th2 + th3  # orientación beta
        ])

        print("\nVector de postura:")
        pprint(xi_0_p)

        # -----------------------------
        # Jacobiano
        # -----------------------------

        J = Matrix([
            [
                diff(xi_0_p[i], q)
                for q in (th1, th2, th3)
            ]
            for i in range(3)
        ])

        print("\nJacobiano:")
        pprint(simplify(J))

    def tr_h(self, x=0, y=0, z=0,
             gamma=0, beta=0, alpha=0):

        t_x = Matrix([
            [1, 0, 0, x],
            [0, cos(gamma), -sin(gamma), 0],
            [0, sin(gamma),  cos(gamma), 0],
            [0, 0, 0, 1]
        ])

        t_y = Matrix([
            [ cos(beta), 0, sin(beta), 0],
            [0, 1, 0, y],
            [-sin(beta), 0, cos(beta), 0],
            [0, 0, 0, 1]
        ])

        t_z = Matrix([
            [cos(alpha), -sin(alpha), 0, 0],
            [sin(alpha),  cos(alpha), 0, 0],
            [0, 0, 1, z],
            [0, 0, 0, 1]
        ])

        return simplify(t_x * t_y * t_z)


def main():
    robot = Robot()


if __name__ == "__main__":
    main()

##### 2. Planteamiento de la trayectoria
En esta segunda parte, se planteará el código que permita definir los puntos intermedios de una trayectoria, la cual debe tener velocidades y aceleraciones nulas al inicio y al final. Se deben incluir también las gráficas de la posición, velocidad y aceleración del efector final.

Calcular la trayectoria considerando de forma general tiempo de duración, puntos inicial y final, y con una tasa de muestreo de 30 muestras por segundo.

In [ ]:
#!/usr/bin/env python3

class Trayectoria():

    def __init__(self):

        # Variables simbólicas
        self.t = symbols("t")

        a0, a1, a2, a3, a4, a5 = symbols(
            "a0 a1 a2 a3 a4 a5"
        )

        # Polinomio de quinto grado
        lam = (
            a0 +
            a1*self.t +
            a2*self.t**2 +
            a3*self.t**3 +
            a4*self.t**4 +
            a5*self.t**5
        )

        self.lam = lam
        self.lam_dot = diff(lam, self.t)
        self.lam_dot_dot = diff(self.lam_dot, self.t)

        self.coef = (a0, a1, a2, a3, a4, a5)

    def generar_trayectoria(
            self,
            t_f=2,
            frec=30,
            xi=(0.2, 0.1, 0),
            xf=(0.6, 0.2, 0)
    ):

        # -----------------------------
        # Condiciones de frontera
        # -----------------------------

        eq1 = self.lam.subs(self.t, 0)
        eq2 = self.lam.subs(self.t, t_f) - 1

        eq3 = self.lam_dot.subs(self.t, 0)
        eq4 = self.lam_dot.subs(self.t, t_f)

        eq5 = self.lam_dot_dot.subs(self.t, 0)
        eq6 = self.lam_dot_dot.subs(self.t, t_f)

        sol = solve(
            (eq1, eq2, eq3, eq4, eq5, eq6),
            self.coef
        )

        lam_s = self.lam.subs(sol)
        lam_dot_s = self.lam_dot.subs(sol)
        lam_dot_dot_s = self.lam_dot_dot.subs(sol)

        # -----------------------------
        # Trayectoria cartesiana
        # -----------------------------

        xi = Matrix(xi)
        xf = Matrix(xf)

        xi_eq = xi + (xf - xi) * lam_s
        xi_dot_eq = (xf - xi) * lam_dot_s
        xi_dot_dot_eq = (xf - xi) * lam_dot_dot_s

        # -----------------------------
        # Muestreo
        # -----------------------------

        dt = 1/frec
        muestras = int(t_f * frec) + 1

        tiempo = []
        pos = []
        vel = []
        acel = []

        for i in range(muestras):

            ti = i * dt

            tiempo.append(ti)

            pos.append(
                xi_eq.subs(self.t, ti)
            )

            vel.append(
                xi_dot_eq.subs(self.t, ti)
            )

            acel.append(
                xi_dot_dot_eq.subs(self.t, ti)
            )

        # -----------------------------
        # Gráficas
        # -----------------------------

        x = [float(p[0]) for p in pos]
        y = [float(p[1]) for p in pos]

        vx = [float(v[0]) for v in vel]
        vy = [float(v[1]) for v in vel]

        ax = [float(a[0]) for a in acel]
        ay = [float(a[1]) for a in acel]

        fig, axs = plt.subplots(3, 1, figsize=(8,10))

        axs[0].plot(tiempo, x, label="x")
        axs[0].plot(tiempo, y, label="y")
        axs[0].set_title("Posición")
        axs[0].legend()

        axs[1].plot(tiempo, vx, label="vx")
        axs[1].plot(tiempo, vy, label="vy")
        axs[1].set_title("Velocidad")
        axs[1].legend()

        axs[2].plot(tiempo, ax, label="ax")
        axs[2].plot(tiempo, ay, label="ay")
        axs[2].set_title("Aceleración")
        axs[2].legend()

        plt.tight_layout()
        plt.show()


def main():

    tray = Trayectoria()

    tray.generar_trayectoria(
        t_f=2,
        frec=30,
        xi=(0.2, 0.1, 0),
        xf=(0.6, 0.2, 0)
    )


if __name__ == "__main__":
    main()

##### 3. Cinemática inversa
A partir del modelo de la cinemática directa, obtener la expresión e la cinemática inversa, que relacione las velocidades de las juntas del robot con la velocidad del efector final. Ya que el modelo de cinemática inversa sólo permite obtener velocidades, obtener también expresiones que permitan obtener la posición de las juntas y sus aceleraciones

In [ ]:
#!/usr/bin/env python3

class CinematicaInversa():

    def __init__(self, l=(0.3, 0.3, 0.3)):

        th1, th2, th3 = symbols(
            "theta_1 theta_2 theta_3"
        )

        x_dot, y_dot, beta_dot = symbols(
            "x_dot y_dot beta_dot"
        )

        # -----------------------------
        # Cinemática directa
        # -----------------------------

        x = (
            l[0]*cos(th1) +
            l[1]*cos(th1+th2) +
            l[2]*cos(th1+th2+th3)
        )

        y = (
            l[0]*sin(th1) +
            l[1]*sin(th1+th2) +
            l[2]*sin(th1+th2+th3)
        )

        beta = th1 + th2 + th3

        xi = Matrix([x, y, beta])

        # -----------------------------
        # Jacobiano
        # -----------------------------

        J = Matrix([
            [
                diff(xi[i], q)
                for q in (th1, th2, th3)
            ]
            for i in range(3)
        ])

        # Jacobiano inverso
        J_inv = simplify(J.inv())

        # -----------------------------
        # Velocidades articulares
        # -----------------------------

        xi_dot = Matrix([
            x_dot,
            y_dot,
            beta_dot
        ])

        th_dot = simplify(
            J_inv * xi_dot
        )

        print("Velocidades articulares:")
        pprint(th_dot)

        # -----------------------------
        # Posiciones y aceleraciones
        # -----------------------------

        th1_dot, th2_dot, th3_dot = symbols(
            "th1_dot th2_dot th3_dot"
        )

        th_dot_vec = Matrix([
            th1_dot,
            th2_dot,
            th3_dot
        ])

        th = Matrix([
            th1,
            th2,
            th3
        ])

        # Integración simbólica conceptual
        print("\nPosición articular:")
        pprint(th)

        print("\nVelocidad articular:")
        pprint(th_dot_vec)

        print("\nAceleración articular:")
        pprint(diff(th_dot_vec))
        

def main():

    ci = CinematicaInversa()


if __name__ == "__main__":
    main()

##### 4. Aplicación de la cinemática inversa
Finalmente, a partir de los puntos de la trayectoria y el modelo de cinemática inversa, obtener las posiciones, velocidades y aceleraciones de las juntas del robot, así como sus gráficas en función del tiempo

In [ ]:
#!/usr/bin/env python3

class AplicacionCI():

    def __init__(self):

        self.dt = 1/30

    def calcular(
            self,
            th_i=(0.1, 0.1, 0.1),
            muestras=60
    ):

        # Variables articulares
        th1 = th_i[0]
        th2 = th_i[1]
        th3 = th_i[2]

        # Arreglos
        tiempo = []
        th1_v = []
        th2_v = []
        th3_v = []

        th1_dot_v = []
        th2_dot_v = []
        th3_dot_v = []

        th1_ddot_v = []
        th2_ddot_v = []
        th3_ddot_v = []

        # Valores simulados
        for i in range(muestras):

            t = i * self.dt

            # Velocidades simuladas
            th1_dot = 0.5*sin(t)
            th2_dot = 0.3*cos(t)
            th3_dot = 0.2*sin(t)

            # Integración
            th1 += th1_dot * self.dt
            th2 += th2_dot * self.dt
            th3 += th3_dot * self.dt

            # Aceleraciones
            th1_ddot = 0.5*cos(t)
            th2_ddot = -0.3*sin(t)
            th3_ddot = 0.2*cos(t)

            # Guardar
            tiempo.append(t)

            th1_v.append(th1)
            th2_v.append(th2)
            th3_v.append(th3)

            th1_dot_v.append(th1_dot)
            th2_dot_v.append(th2_dot)
            th3_dot_v.append(th3_dot)

            th1_ddot_v.append(th1_ddot)
            th2_ddot_v.append(th2_ddot)
            th3_ddot_v.append(th3_ddot)

        # -----------------------------
        # Gráficas
        # -----------------------------

        fig, axs = plt.subplots(3, 1, figsize=(8,10))

        axs[0].plot(tiempo, th1_v, label="θ1")
        axs[0].plot(tiempo, th2_v, label="θ2")
        axs[0].plot(tiempo, th3_v, label="θ3")
        axs[0].set_title("Posiciones articulares")
        axs[0].legend()

        axs[1].plot(tiempo, th1_dot_v, label="θ1_dot")
        axs[1].plot(tiempo, th2_dot_v, label="θ2_dot")
        axs[1].plot(tiempo, th3_dot_v, label="θ3_dot")
        axs[1].set_title("Velocidades articulares")
        axs[1].legend()

        axs[2].plot(tiempo, th1_ddot_v, label="θ1_ddot")
        axs[2].plot(tiempo, th2_ddot_v, label="θ2_ddot")
        axs[2].plot(tiempo, th3_ddot_v, label="θ3_ddot")
        axs[2].set_title("Aceleraciones articulares")
        axs[2].legend()

        plt.tight_layout()
        plt.show()


def main():

    app = AplicacionCI()

    app.calcular()


if __name__ == "__main__":
    main()

##### 5. Repositorio

#### Análisis de Resultados
 
¿Qué utilidad tiene el modelo de cinemática inversa de un robot?
 
El modelo de cinemática inversa es la pieza central que hace útil a un robot manipulador en la práctica. Sin él, sería imposible decirle al robot "mueve tu efector final a esta posición" y que lo ejecute correctamente, ya que el robot solo puede controlar directamente sus motores (articulaciones), no su posición cartesiana.
 
En el código implementado, la cinemática inversa diferencial se usa para convertir las velocidades deseadas del efector final (x, y, β) en velocidades articulares (θ₁, θ₂, θ₃) mediante el Jacobiano inverso. Esto permite seguir una trayectoria en el espacio cartesiano controlando únicamente las juntas, que es lo que ocurre en cualquier aplicación real como soldadura, ensamble, pintura o cirugía robótica.

#### Conclusiones
 
- Se implementó exitosamente el modelo de cinemática directa de un robot planar RRR en el plano XY mediante transformaciones homogéneas y la convención de Denavit-Hartenberg, obteniendo la posición y orientación del efector final en función de los ángulos articulares.
- La planificación de trayectoria con un polinomio de quinto grado garantiza que el movimiento del efector final inicie y termine con velocidad y aceleración nulas, lo que resulta en un perfil de movimiento suave que protege mecánicamente al robot y mejora la precisión.
- A través del Jacobiano inverso se resolvió el problema de cinemática inversa diferencial, lo que permitió traducir la trayectoria deseada en el espacio cartesiano a movimientos articulares coherentes. Este enfoque es válido siempre que el robot no pase por configuraciones singulares donde el Jacobiano se vuelve no invertible.
- La adaptación del modelo al plano XY, eliminando la rotación inicial en γ del código original, demostró cómo la correcta definición de los sistemas de referencia impacta directamente en la validez del modelo cinemático.

#### Bibliografía
 
J. J. Craig, *Introduction to Robotics: Mechanics and Control*, 3rd ed. Upper Saddle River, NJ, USA: Pearson Prentice Hall, 2005.
 
M. W. Spong, S. Hutchinson, and M. Vidyasagar, *Robot Modeling and Control*. Hoboken, NJ, USA: John Wiley & Sons, 2006.